In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import joblib

pd.set_option("display.max_columns", 200)


In [2]:

csv_path = "amazon_laptop_prices_v01 (1).csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)

display(df.head())

Shape: (4446, 17)


,brand,model,screen_size,color,harddisk,cpu,ram,OS,special_features,graphics,graphics_coprocessor,cpu_speed,rating,Price,Sale Product Count,Total Sales,Available Stock
0,ROKC,NaN,14 Inches,Blue,1000 GB,Intel Core i7,8 GB,Windows 11,NaN,Integrated,Intel,1.2 GHz,NaN,"$1,783.99",14,24975.86,81.0
1,HP,NaN,15.6 Inches,Silver,1000 GB,Intel Core i5,64 GB,Windows 11 Pro,Backlit Keyboard,Integrated,Intel,NaN,4.5,$449.00,60,26940,334.0
2,MSI,Vector GP66 12UGS-267,15.66 Inches,Core Black,NaN,Intel Core i9,32 GB,Windows 11 Home,NaN,Dedicated,NaN,1.8 GHz,5.0,$999.99,24,23999.76,391.0
3,Apple,MacBook Air,13.3 Inches,Silver,256 GB,Unknown,8 GB,Mac OS,Backlit Keyboard,Integrated,NaN,NaN,4.8,"$3,423.99",60,205439.4,345.0
4,Apple,MacBook Air,15.3 Inches,Midnight,256 GB,Unknown,8 GB,Mac OS,NaN,Integrated,NaN,NaN,4.8,$699.00,33,23067,110.0


In [3]:

display(df.describe(include="all").T.head(25))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
brand,4446,50,Dell,1581,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,3282,1083,Vector GP66 12UGS-267,196,NaN,NaN,NaN,NaN,NaN,NaN,NaN
screen_size,4417,35,15.6 Inches,1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
color,3867,187,Black,1172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
harddisk,3870,41,1000 GB,1334,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpu,4346,141,Core i7,828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ram,4385,18,16 GB,1344,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OS,4420,49,Windows 11,1199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
special_features,2054,188,Wifi & Bluetooth,815,NaN,NaN,NaN,NaN,NaN,NaN,NaN
graphics,4381,109,Integrated,3021,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:


def to_float(x):
    """Best-effort conversion to float."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x)
    s = s.replace(",", "")
    # keep digits and dot only
    m = re.findall(r"\d+\.?\d*", s)
    if not m:
        return np.nan
    try:
        return float(m[0])
    except:
        return np.nan

def parse_price(x):
    """Extract numeric price from strings like '$1,299' or '1299'."""
    return to_float(x)

def parse_ram_gb(x):
    """Extract RAM in GB from strings like '16GB' '8 GB' '32gb DDR4'."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    # prefer GB
    m = re.search(r"(\d+)\s*gb", s)
    if m:
        return float(m.group(1))
    # sometimes MB
    m = re.search(r"(\d+)\s*mb", s)
    if m:
        return float(m.group(1)) / 1024.0
    return np.nan

def parse_storage_gb(x):
    """Extract storage from strings like '512GB SSD', '1TB HDD', '256 GB'"""
    if pd.isna(x):
        return np.nan
    s = str(x).lower().replace(",", " ")
    # capture TB first
    m = re.search(r"(\d+(?:\.\d+)?)\s*tb", s)
    if m:
        return float(m.group(1)) * 1024.0
    # then GB
    m = re.search(r"(\d+(?:\.\d+)?)\s*gb", s)
    if m:
        return float(m.group(1))
    return np.nan

def cpu_tier(cpu_text):
    """Map CPU description to a rough tier score."""
    if pd.isna(cpu_text):
        return np.nan
    s = str(cpu_text).lower()

    # Intel
    if "ultra" in s and "intel" in s:
        return 6
    if "i9" in s:
        return 6
    if "i7" in s:
        return 5
    if "i5" in s:
        return 4
    if "i3" in s:
        return 3
    if "pentium" in s or "celeron" in s:
        return 1

    # AMD
    if "ryzen 9" in s:
        return 6
    if "ryzen 7" in s:
        return 5
    if "ryzen 5" in s:
        return 4
    if "ryzen 3" in s:
        return 3
    if "athlon" in s:
        return 1

    # Apple
    if "m3" in s:
        return 6
    if "m2" in s:
        return 5
    if "m1" in s:
        return 4

    return np.nan

def is_ssd(text):
    if pd.isna(text):
        return 0
    return int("ssd" in str(text).lower())

def has_dedicated_gpu(text):
    if pd.isna(text):
        return 0
    s = str(text).lower()
    keywords = ["nvidia", "rtx", "gtx", "radeon", "rx "]
    return int(any(k in s for k in keywords))



In [5]:
# -----------------------------
# 2) Apply cleaning + features
# -----------------------------
df2 = df.copy()

# Standardize column names a bit
df2.columns = [c.strip() for c in df2.columns]

# Price
df2["price"] = df2["Price"].apply(parse_price)

# Numeric conversions
df2["screen_size_in"] = df2["screen_size"].apply(to_float)
df2["cpu_speed_ghz"] = df2["cpu_speed"].apply(to_float)
df2["rating_num"] = df2["rating"].apply(to_float)

# Extract RAM + storage
df2["ram_gb"] = df2["ram"].apply(parse_ram_gb)
df2["storage_gb"] = df2["harddisk"].apply(parse_storage_gb)

# CPU tier
df2["cpu_tier"] = df2["cpu"].apply(cpu_tier)

# Flags
df2["is_ssd"] = df2["harddisk"].apply(is_ssd)
df2["dedicated_gpu"] = df2["graphics"].apply(has_dedicated_gpu)

# Drop impossible prices
df2 = df2[(df2["price"].isna()) | (df2["price"] > 0)]

print("After basic cleaning:", df2.shape)
display(df2[["brand","model","price","ram","ram_gb","harddisk","storage_gb","cpu","cpu_tier","graphics","dedicated_gpu","rating_num"]].head(10))


After basic cleaning: (4446, 26)


,brand,model,price,ram,ram_gb,harddisk,storage_gb,cpu,cpu_tier,graphics,dedicated_gpu,rating_num
0,ROKC,NaN,1783.99,8 GB,8.0,1000 GB,1000.0,Intel Core i7,5.0,Integrated,0,NaN
1,HP,NaN,449.00,64 GB,64.0,1000 GB,1000.0,Intel Core i5,4.0,Integrated,0,4.5
2,MSI,Vector GP66 12UGS-267,999.99,32 GB,32.0,NaN,NaN,Intel Core i9,6.0,Dedicated,0,5.0
3,Apple,MacBook Air,3423.99,8 GB,8.0,256 GB,256.0,Unknown,NaN,Integrated,0,4.8
4,Apple,MacBook Air,699.00,8 GB,8.0,256 GB,256.0,Unknown,NaN,Integrated,0,4.8
5,Acer,A315-24P-R7VH,1599.00,8 GB,8.0,128 GB,128.0,Ryzen 3,3.0,Integrated,0,4.5
6,Apple,MacBook Pro,1314.99,8 GB,8.0,256 GB,256.0,Unknown,NaN,Integrated,0,4.7
7,Acer,CB315-3HT,799.00,4 GB,4.0,64 GB,64.0,Celeron N4020,1.0,Integrated,0,4.4
8,ASUS,ROG Strix G16,899.99,16 GB,16.0,512 GB,512.0,Core i7,5.0,Dedicated,0,4.4
9,acer,A515-56-347N,1037.99,8 GB,8.0,128 GB,128.0,Core I3 1115G4,3.0,Integrated,0,4.3


In [6]:
# ---------------------------------------
# 3) Feature Engineering (Value Score)
# ---------------------------------------
# A simple, explainable performance score. You can refine weights later.
# - RAM is usually very important for general performance
# - CPU tier captures processor strength
# - SSD improves responsiveness
# - Dedicated GPU is important for gaming/ML/video
# - Storage helps but with smaller weight

df2["performance_score"] = (
    2.0 * df2["ram_gb"].fillna(df2["ram_gb"].median()) +
    1.5 * df2["cpu_tier"].fillna(df2["cpu_tier"].median()) +
    0.002 * df2["storage_gb"].fillna(df2["storage_gb"].median()) +
    1.0 * df2["is_ssd"].fillna(0) +
    1.5 * df2["dedicated_gpu"].fillna(0)
)

# Value-for-money: higher is better
df2["value_score"] = df2["performance_score"] / df2["price"]

# Some rows may have missing price; remove those for value-based labeling
df_value = df2.dropna(subset=["brand","price","value_score"]).copy()

# Avoid tiny-sample brands dominating (data scientist practice)
min_brand_count = 20
brand_counts = df_value["brand"].value_counts()
valid_brands = brand_counts[brand_counts >= min_brand_count].index
df_value = df_value[df_value["brand"].isin(valid_brands)].copy()

brand_value = df_value.groupby("brand")["value_score"].mean().sort_values(ascending=False)
best_brand = brand_value.index[0]

print("Best brand by avg value_score (min count =", min_brand_count, "):", best_brand)
display(brand_value.head(15))

df_value["is_best_brand"] = (df_value["brand"] == best_brand).astype(int)
df_value["is_best_brand"].value_counts().rename({0:"Other brands",1:"Best brand"})


Best brand by avg value_score (min count = 20 ): DELL


,value_score
brand,
DELL,0.091228
HP,0.089338
MSI,0.085565
Dell,0.074163
Apple,0.068376
LG,0.061475
ROKC,0.054499
Lenovo,0.045677
ASUS,0.042893


,count
is_best_brand,
Other brands,4162
Best brand,147


In [7]:
df_value.columns

Index(['brand', 'model', 'screen_size', 'color', 'harddisk', 'cpu', 'ram',
       'OS', 'special_features', 'graphics', 'graphics_coprocessor',
       'cpu_speed', 'rating', 'Price', 'Sale Product Count', 'Total Sales',
       'Available Stock', 'price', 'screen_size_in', 'cpu_speed_ghz',
       'rating_num', 'ram_gb', 'storage_gb', 'cpu_tier', 'is_ssd',
       'dedicated_gpu', 'performance_score', 'value_score', 'is_best_brand'],
      dtype='object')

In [8]:
df_value['brand'].value_counts()

,count
brand,
Dell,1581
HP,804
ROKC,637
MSI,473
Lenovo,308
ASUS,160
DELL,147
acer,108
LG,39


In [9]:
df=df_value

In [10]:
df.isnull().sum()

,0
brand,0
model,1152
screen_size,25
color,542
harddisk,562
cpu,87
ram,60
OS,23
special_features,2332
graphics,63


In [11]:
df['brand'].value_counts()

,count
brand,
Dell,1581
HP,804
ROKC,637
MSI,473
Lenovo,308
ASUS,160
DELL,147
acer,108
LG,39


In [12]:
df['brand'] = (
    df['brand']
    .str.strip()
    .str.upper()
)


In [13]:
brand_map = {
    "DELL": "DELL",
    "HP": "HP",
    "ACER": "ACER",
    "LENOVO": "LENOVO",
    "ASUS": "ASUS",
    "APPLE": "APPLE",
    "SAMSUNG": "SAMSUNG",
    "MSI": "MSI",
    "LG": "LG",
    "ROKC": "ROKC"
}

df['brand'] = df['brand'].map(brand_map).fillna(df['brand'])


In [14]:
df['brand'].value_counts()


,count
brand,
DELL,1728
HP,804
ROKC,637
MSI,473
LENOVO,308
ASUS,160
ACER,108
LG,39
APPLE,29


In [15]:
import pandas as pd

target_col = "brand"

max_n = df[target_col].value_counts().max()

df = (
    df.groupby(target_col, group_keys=False)
      .apply(lambda g: g.sample(n=max_n, replace=True, random_state=42))
      .reset_index(drop=True)
)

print("Before:\n", df[target_col].value_counts())
print("\nAfter:\n", df[target_col].value_counts())



Before:
 brand
ACER       1728
APPLE      1728
ASUS       1728
DELL       1728
HP         1728
LENOVO     1728
LG         1728
MSI        1728
ROKC       1728
SAMSUNG    1728
Name: count, dtype: int64

After:
 brand
ACER       1728
APPLE      1728
ASUS       1728
DELL       1728
HP         1728
LENOVO     1728
LG         1728
MSI        1728
ROKC       1728
SAMSUNG    1728
Name: count, dtype: int64


/tmp/ipython-input-1364990657.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max_n, replace=True, random_state=42))


In [16]:
df.drop(columns=['model',])

,brand,screen_size,color,harddisk,cpu,ram,OS,special_features,graphics,graphics_coprocessor,cpu_speed,rating,Price,Sale Product Count,Total Sales,Available Stock,price,screen_size_in,cpu_speed_ghz,rating_num,ram_gb,storage_gb,cpu_tier,is_ssd,dedicated_gpu,performance_score,value_score,is_best_brand
0,ACER,17.3 Inches,Obsidian Black,4 TB,Ryzen 7,64 GB,Windows 10 Pro,"HD Audio, Backlit Keyboard, Anti Glare Coating...",Dedicated,NaN,NaN,NaN,"$3,431.25",50,171562.5,443.0,3431.25,17.3,NaN,NaN,64.0,4096.0,5.0,0,0,143.692,0.041877,0
1,ACER,11.6 Inches,Black,32 GB,MediaTek MT8183,4 GB,Chrome OS,NaN,Integrated,Intel Integrated Graphics,NaN,4.4,$719.99,61,43919.39,204.0,719.99,11.6,NaN,4.4,4.0,32.0,NaN,0,0,15.564,0.021617,0
2,ACER,14 Inches,NaN,256 GB,Core i5 Family,16 GB,Windows 10 Pro,Anti-glare Screen,Iris Xe Graphics,Intel Iris Xe Graphics,NaN,NaN,$459.99,25,11499.75,262.0,459.99,14.0,NaN,NaN,16.0,256.0,4.0,0,0,38.512,0.083724,0
3,ACER,14 Inches,Gray,512 GB,Core i5,8 GB,Windows 11 Home,Backlit Keyboard,Integrated,NaN,NaN,NaN,$951.99,39,37127.61,134.0,951.99,14.0,NaN,NaN,8.0,512.0,4.0,0,0,23.024,0.024185,0
4,ACER,NaN,NaN,NaN,NaN,16 GB,"Windows 11 Pro, Windows",NaN,Integrated,NaN,NaN,NaN,$696.34,27,38759.49,NaN,696.34,NaN,NaN,NaN,16.0,NaN,NaN,0,0,41.500,0.059597,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17275,SAMSUNG,15.6 Inches,Silver,16 GB,Core i7,16 GB,Windows 11 Pro,NaN,Integrated,Intel Iris Xe Graphics,NaN,4.8,$732.00,60,43920,231.0,732.00,15.6,NaN,4.8,16.0,16.0,5.0,0,0,39.532,0.054005,0
17276,SAMSUNG,12.2 Inches,Light Titan,32 GB,Celeron,4 GB,Chrome OS,Pen,Integrated,NaN,NaN,4.5,"$1,109.18",35,38821.3,400.0,1109.18,12.2,NaN,4.5,4.0,32.0,1.0,0,0,9.564,0.008623,0
17277,SAMSUNG,11.6 Inches,Silver,64 GB,Celeron N3450,4 GB,Chrome OS,NaN,Integrated,Intel UHD Graphics 600,NaN,4.9,$229.99,25,5749.75,410.0,229.99,11.6,NaN,4.9,4.0,64.0,1.0,0,0,9.628,0.041863,0
17278,SAMSUNG,14 Inches,Graphite,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.4,$999.99,18,17999.82,121.0,999.99,14.0,NaN,4.4,NaN,NaN,NaN,0,0,41.500,0.041500,0


In [17]:
df.columns

Index(['brand', 'model', 'screen_size', 'color', 'harddisk', 'cpu', 'ram',
       'OS', 'special_features', 'graphics', 'graphics_coprocessor',
       'cpu_speed', 'rating', 'Price', 'Sale Product Count', 'Total Sales',
       'Available Stock', 'price', 'screen_size_in', 'cpu_speed_ghz',
       'rating_num', 'ram_gb', 'storage_gb', 'cpu_tier', 'is_ssd',
       'dedicated_gpu', 'performance_score', 'value_score', 'is_best_brand'],
      dtype='object')

In [18]:
cols_to_drop = [
    # target
    "brand",

    # id / leakage
    "model",

    # raw / duplicated
    "screen_size",
    "cpu_speed",
    "rating",
    "ram",
    "Price",

    # derived labels
    "value_score",
    "is_best_brand",

    # sales / inventory
    "Sale Product Count",
    "Total Sales",
    "Available Stock"
]

X = df.drop(columns=cols_to_drop)
y = df["brand"]


In [19]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(X.columns)


Features shape: (17280, 17)
Target shape: (17280,)
Index(['color', 'harddisk', 'cpu', 'OS', 'special_features', 'graphics',
       'graphics_coprocessor', 'price', 'screen_size_in', 'cpu_speed_ghz',
       'rating_num', 'ram_gb', 'storage_gb', 'cpu_tier', 'is_ssd',
       'dedicated_gpu', 'performance_score'],
      dtype='object')


In [20]:
X.isnull().sum()


,0
color,2916
harddisk,2674
cpu,1100
OS,211
special_features,9370
graphics,296
graphics_coprocessor,8641
price,0
screen_size_in,413
cpu_speed_ghz,11698


In [21]:
num_cols = [
    "price","screen_size_in","cpu_speed_ghz","rating_num",
    "ram_gb","storage_gb","cpu_tier","is_ssd",
    "dedicated_gpu","performance_score"
]

cat_cols = [
    "color","harddisk","cpu","OS","special_features",
    "graphics","graphics_coprocessor"
]


In [22]:
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].mode()[0])


In [23]:
for col in num_cols:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].median())


In [24]:
X.head()

,color,harddisk,cpu,OS,special_features,graphics,graphics_coprocessor,price,screen_size_in,cpu_speed_ghz,rating_num,ram_gb,storage_gb,cpu_tier,is_ssd,dedicated_gpu,performance_score
0,Obsidian Black,4 TB,Ryzen 7,Windows 10 Pro,"HD Audio, Backlit Keyboard, Anti Glare Coating...",Dedicated,Intel,3431.25,17.3,1.8,4.4,64.0,4096.0,5.0,0,0,143.692
1,Black,32 GB,MediaTek MT8183,Chrome OS,Backlit Keyboard,Integrated,Intel Integrated Graphics,719.99,11.6,1.8,4.4,4.0,32.0,5.0,0,0,15.564
2,Black,256 GB,Core i5 Family,Windows 10 Pro,Anti-glare Screen,Iris Xe Graphics,Intel Iris Xe Graphics,459.99,14.0,1.8,4.4,16.0,256.0,4.0,0,0,38.512
3,Gray,512 GB,Core i5,Windows 11 Home,Backlit Keyboard,Integrated,Intel,951.99,14.0,1.8,4.4,8.0,512.0,4.0,0,0,23.024
4,Black,512 GB,Core i7,"Windows 11 Pro, Windows",Backlit Keyboard,Integrated,Intel,696.34,15.6,1.8,4.4,16.0,512.0,5.0,0,0,41.500


In [25]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le


In [26]:
brand_le = LabelEncoder()
y_enc = brand_le.fit_transform(y.astype(str).str.strip().str.upper())


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=brand_le.classes_))


Accuracy: 0.9918981481481481
              precision    recall  f1-score   support

        ACER       0.99      1.00      1.00       345
       APPLE       1.00      1.00      1.00       345
        ASUS       0.98      1.00      0.99       346
        DELL       0.99      0.96      0.97       346
          HP       1.00      0.97      0.98       346
      LENOVO       0.96      1.00      0.98       345
          LG       1.00      1.00      1.00       346
         MSI       1.00      0.99      0.99       346
        ROKC       1.00      1.00      1.00       345
     SAMSUNG       1.00      1.00      1.00       346

    accuracy                           0.99      3456
   macro avg       0.99      0.99      0.99      3456
weighted avg       0.99      0.99      0.99      3456



In [28]:
pred_encoded = model.predict(X_test.iloc[:5])
pred_brand = brand_le.inverse_transform(pred_encoded)

print("Predicted brands:", pred_brand)


Predicted brands: ['APPLE' 'DELL' 'HP' 'HP' 'ASUS']


In [29]:
new_row = X_test.iloc[[0]].copy()

pred_new_enc = model.predict(new_row)[0]
pred_new_brand = brand_le.inverse_transform([pred_new_enc])[0]

print("Predicted brand:", pred_new_brand)


Predicted brand: APPLE


In [30]:
X=X.drop(columns=['OS'])

In [31]:
X.head()

,color,harddisk,cpu,special_features,graphics,graphics_coprocessor,price,screen_size_in,cpu_speed_ghz,rating_num,ram_gb,storage_gb,cpu_tier,is_ssd,dedicated_gpu,performance_score
0,102,20,107,116,6,38,3431.25,17.3,1.8,4.4,64.0,4096.0,5.0,0,0,143.692
1,15,17,89,48,17,66,719.99,11.6,1.8,4.4,4.0,32.0,5.0,0,0,15.564
2,15,14,45,15,29,71,459.99,14.0,1.8,4.4,16.0,256.0,4.0,0,0,38.512
3,60,23,39,48,17,38,951.99,14.0,1.8,4.4,8.0,512.0,4.0,0,0,23.024
4,15,23,52,48,17,38,696.34,15.6,1.8,4.4,16.0,512.0,5.0,0,0,41.500


this is high because my model is cheacting becase of os operating system reveleing their information


again the same issue the cpu is revleaing the information

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=brand_le.classes_))

Accuracy: 0.9921875
              precision    recall  f1-score   support

        ACER       1.00      1.00      1.00       345
       APPLE       1.00      1.00      1.00       345
        ASUS       0.99      1.00      0.99       346
        DELL       0.99      0.96      0.97       346
          HP       0.99      0.97      0.98       346
      LENOVO       0.96      1.00      0.98       345
          LG       1.00      1.00      1.00       346
         MSI       1.00      0.99      0.99       346
        ROKC       1.00      1.00      1.00       345
     SAMSUNG       1.00      1.00      1.00       346

    accuracy                           0.99      3456
   macro avg       0.99      0.99      0.99      3456
weighted avg       0.99      0.99      0.99      3456



In [33]:
pred_encoded = model.predict(X_test.iloc[:5])
pred_brand = brand_le.inverse_transform(pred_encoded)

print("Predicted brands:", pred_brand)


Predicted brands: ['APPLE' 'DELL' 'HP' 'HP' 'ASUS']


In [34]:
X=X.drop(columns=['cpu'])

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=brand_le.classes_))

Accuracy: 0.9927662037037037
              precision    recall  f1-score   support

        ACER       1.00      1.00      1.00       345
       APPLE       1.00      1.00      1.00       345
        ASUS       0.99      1.00      1.00       346
        DELL       0.99      0.96      0.98       346
          HP       0.99      0.97      0.98       346
      LENOVO       0.96      1.00      0.98       345
          LG       1.00      1.00      1.00       346
         MSI       1.00      0.99      0.99       346
        ROKC       1.00      1.00      1.00       345
     SAMSUNG       1.00      1.00      1.00       346

    accuracy                           0.99      3456
   macro avg       0.99      0.99      0.99      3456
weighted avg       0.99      0.99      0.99      3456



In [36]:
ideal_features = [
    "price",
    "screen_size_in",
    "ram_gb",
    "storage_gb",
    "cpu_tier",
    "dedicated_gpu",
    "is_ssd",
    "performance_score"
]


In [37]:
X = X[ideal_features].copy()

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=brand_le.classes_))

Accuracy: 0.9597800925925926
              precision    recall  f1-score   support

        ACER       0.92      1.00      0.96       345
       APPLE       0.99      1.00      1.00       345
        ASUS       0.96      0.95      0.96       346
        DELL       0.94      0.86      0.90       346
          HP       0.97      0.94      0.96       346
      LENOVO       0.91      0.93      0.92       345
          LG       0.98      0.95      0.96       346
         MSI       1.00      0.97      0.98       346
        ROKC       1.00      1.00      1.00       345
     SAMSUNG       0.93      1.00      0.97       346

    accuracy                           0.96      3456
   macro avg       0.96      0.96      0.96      3456
weighted avg       0.96      0.96      0.96      3456



In [39]:
import joblib
joblib.dump(model, "brand_prediction_model.joblib")
joblib.dump(brand_le, "brand_label_encoder.joblib")

print("✅ Model and encoder exported successfully")


✅ Model and encoder exported successfully


In [41]:
df.isnull().sum()

,0
brand,0
model,3639
screen_size,413
color,2916
harddisk,2674
cpu,1100
ram,308
OS,211
special_features,9370
graphics,296
